# Day 8: Batching — Processing Multiple Samples

**Learning Objective**: Implement batch processing to train on multiple samples simultaneously, improving training efficiency and gradient estimates.

Yesterday we trained on all samples every step. Today we learn **mini-batch gradient descent**:
- **Faster**: Process subsets of data
- **Smoother**: Average gradients reduce noise
- **GPU-friendly**: Modern hardware loves batches

In [ ]:
import math
import random

## Theory: Three Types of Gradient Descent

| Method | Batch Size | Pros | Cons |
|--------|------------|------|------|
| **Batch GD** | All data | Stable gradients | Slow, memory-heavy |
| **Stochastic GD** | 1 sample | Fast updates | Noisy gradients |
| **Mini-batch GD** | 16-256 | Best of both | Needs tuning |

### Why Batching Works
The mini-batch gradient is an **unbiased estimator** of the true gradient:

$$\nabla L \approx \frac{1}{B}\sum_{i=1}^{B} \nabla L_i$$

Larger batch → lower variance → smoother training.

## Neural Network Code (from Week 1)

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) - self
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return Value(other) * self**-1

In [ ]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0)
    
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()
    
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

## MSE Loss Function

In [ ]:
def mse_loss(predictions, targets):
    """Mean Squared Error loss."""
    n = len(predictions)
    return sum((p - t) ** 2 for p, t in zip(predictions, targets)) * (1.0 / n)

## Create Batches

Split the dataset into mini-batches with shuffling.

In [ ]:
def create_batches(X, y, batch_size):
    """
    Create mini-batches from dataset with shuffling.
    
    Args:
        X: List of input samples
        y: List of target values
        batch_size: Number of samples per batch
    
    Returns:
        List of (X_batch, y_batch) tuples
    """
    n = len(X)
    indices = list(range(n))
    random.shuffle(indices)
    
    batches = []
    for i in range(0, n, batch_size):
        batch_idx = indices[i:i+batch_size]
        X_batch = [X[j] for j in batch_idx]
        y_batch = [y[j] for j in batch_idx]
        batches.append((X_batch, y_batch))
    
    return batches

## Test 1: Batch Creation

In [ ]:
X_test = [[i] for i in range(10)]
y_test = list(range(10))

batches = create_batches(X_test, y_test, batch_size=3)

print(f"Number of batches: {len(batches)}")
for i, (xb, yb) in enumerate(batches):
    print(f"  Batch {i}: X={[x[0] for x in xb]}, y={yb}")

# Verify all samples are covered
all_x = []
for xb, yb in batches:
    all_x.extend([x[0] for x in xb])
assert sorted(all_x) == list(range(10)), "Not all samples covered!"
assert len(batches) == 4, "Should have 4 batches (3+3+3+1)"
print("\n✅ Batch creation works!")

## Batched Training Loop

In [ ]:
def train_epoch_batched(model, batches, lr):
    """Train for one epoch using pre-created batches."""
    total_loss = 0.0
    
    for X_batch, y_batch in batches:
        # Forward pass
        preds = [model(x) for x in X_batch]
        
        # Compute loss
        loss = mse_loss(preds, y_batch)
        total_loss += loss.data
        
        # Zero gradients
        for p in model.parameters():
            p.grad = 0.0
        
        # Backward pass
        loss.backward()
        
        # Update parameters
        for p in model.parameters():
            p.data -= lr * p.grad
    
    return total_loss / len(batches)


def train_batched(model, X, y, epochs=100, lr=0.01, batch_size=4, verbose=True):
    """Full training loop with mini-batches."""
    losses = []
    
    for epoch in range(epochs):
        # Create new shuffled batches each epoch
        batches = create_batches(X, y, batch_size)
        
        # Train one epoch
        avg_loss = train_epoch_batched(model, batches, lr)
        losses.append(avg_loss)
        
        if verbose and epoch % 20 == 0:
            print(f"Epoch {epoch:4d}: loss = {avg_loss:.6f}")
    
    return losses

## Test 2: Batched Training Reduces Loss

In [ ]:
random.seed(42)

# Create dataset: learn y = 2*x1 - x2
X = [[random.uniform(-1, 1), random.uniform(-1, 1)] for _ in range(50)]
y = [x[0] * 2 - x[1] for x in X]

# Create model
model = MLP(2, [8, 1])

# Train with batches
losses = train_batched(model, X, y, epochs=100, lr=0.05, batch_size=8)

print(f"\nLoss: {losses[0]:.4f} → {losses[-1]:.4f}")
assert losses[-1] < losses[0], "Loss should decrease!"
print("✅ Batched training works!")

## Experiment: Compare Batch Sizes

See how different batch sizes affect training.

In [ ]:
# Create a larger dataset
random.seed(123)
X = [[random.uniform(-1, 1), random.uniform(-1, 1)] for _ in range(100)]
y = [x[0] + x[1] for x in X]

print("Batch Size Comparison:")
print("=" * 50)

for batch_size in [1, 4, 16, 50, 100]:
    random.seed(42)
    model = MLP(2, [4, 1])
    losses = train_batched(model, X, y, epochs=50, lr=0.1,
                           batch_size=batch_size, verbose=False)
    
    status = "✅" if losses[-1] < 0.1 else "⚠️" if losses[-1] < 1.0 else "❌"
    print(f"  batch_size={batch_size:3d}: final_loss={losses[-1]:.6f} {status}")

print("\nObservations:")
print("  - batch_size=1 (SGD): Noisy but fast updates")
print("  - batch_size=100 (full batch): Stable but slow per-epoch")
print("  - batch_size=4-16: Best balance of speed and stability")

## Experiment: SGD vs Full Batch — Loss Trajectory

In [ ]:
# Track loss over time for SGD (batch=1) vs full batch
random.seed(42)
X = [[random.uniform(-1, 1), random.uniform(-1, 1)] for _ in range(50)]
y = [x[0] * 2 - x[1] for x in X]

# SGD (batch_size=1)
random.seed(42)
model_sgd = MLP(2, [4, 1])
losses_sgd = train_batched(model_sgd, X, y, epochs=100, lr=0.05,
                           batch_size=1, verbose=False)

# Mini-batch (batch_size=8)
random.seed(42)
model_mini = MLP(2, [4, 1])
losses_mini = train_batched(model_mini, X, y, epochs=100, lr=0.05,
                            batch_size=8, verbose=False)

# Full batch
random.seed(42)
model_full = MLP(2, [4, 1])
losses_full = train_batched(model_full, X, y, epochs=100, lr=0.05,
                            batch_size=50, verbose=False)

print("Loss Trajectory (every 20 epochs):")
print(f"{'Epoch':>6} | {'SGD (bs=1)':>12} | {'Mini (bs=8)':>12} | {'Full (bs=50)':>12}")
print("-" * 52)
for i in range(0, 100, 20):
    print(f"{i:6d} | {losses_sgd[i]:12.6f} | {losses_mini[i]:12.6f} | {losses_full[i]:12.6f}")
print(f"{'Final':>6} | {losses_sgd[-1]:12.6f} | {losses_mini[-1]:12.6f} | {losses_full[-1]:12.6f}")

## Summary

Today we implemented:

1. **`create_batches()`**: Shuffles data and splits into mini-batches
2. **`train_epoch_batched()`**: One epoch of mini-batch training
3. **`train_batched()`**: Full training loop with configurable batch size

**Key takeaways:**
- Mini-batch GD balances gradient noise and computation
- Shuffling each epoch prevents the model from memorizing order
- Batch size 4-32 is typically a good starting point
- Always zero gradients before each backward pass!

---

*Previous: [Day 7 — Training Loop](./day_07_training_loop.ipynb)*  
*Next: Day 9 — Momentum*